# Step 1: Embedding-Based Anomaly Score on NYC Taxi (NAB)

**TimesFM 2.5 API**: `TimesFM_2p5_200M_torch.from_pretrained(...).compile(ForecastConfig(...))`

**Method:** for each window, get the hidden state from a middle layer of TimesFM 2.5, then score by Mahalanobis distance to the cloud of normal embeddings.

References: THEMIS (arXiv 2510.03911), TimeRep (arXiv 2509.12650), NAB (Lavin & Ahmad 2015).


## 1. Install

`pip install timesfm` may give you the old 1.x/2.0 API. Install from GitHub for 2.5:


In [ ]:
# !pip install -U "git+https://github.com/google-research/timesfm.git#egg=timesfm[torch]"
# !pip install -U kagglehub pandas matplotlib scikit-learn


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
import timesfm

assert torch.cuda.is_available(), "GPU required for TimesFM 2.5"
DEVICE = "cuda"
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)

# Sanity check the install — should print the 2.5 class name
assert hasattr(timesfm, "TimesFM_2p5_200M_torch"), \
    "Your timesfm install is too old. Install from GitHub: pip install -U git+https://github.com/google-research/timesfm.git"
print("timesfm OK | cuda:", torch.cuda.get_device_name(0))


## 2. Load NYC Taxi data + NAB labels

NAB anomaly windows for nyc_taxi.csv are public — every timestamp inside one of these five windows is `is_true_anomaly = True`.


In [ ]:
NAB_ANOMALY_WINDOWS = [
    ("2014-10-30 15:30:00", "2014-11-03 22:30:00"),   # NYC Marathon
    ("2014-11-25 12:00:00", "2014-11-29 19:00:00"),   # Thanksgiving
    ("2014-12-23 11:30:00", "2014-12-27 18:30:00"),   # Christmas
    ("2014-12-29 21:30:00", "2015-01-03 04:30:00"),   # New Year's Day
    ("2015-01-24 20:30:00", "2015-01-29 03:30:00"),   # Snow Storm
]

def load_nyc_taxi(csv_path: str | None = None) -> pd.DataFrame:
    if csv_path is None:
        import kagglehub, os
        root = kagglehub.dataset_download("julienjta/nyc-taxi-traffic")
        for fn in os.listdir(root):
            if fn.endswith(".csv"):
                csv_path = os.path.join(root, fn); break
        assert csv_path, f"no csv found in {root}"

    df = pd.read_csv(csv_path)
    df.columns = [c.strip().lower() for c in df.columns]
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp").reset_index(drop=True)

    is_anom = np.zeros(len(df), dtype=bool)
    for s, e in NAB_ANOMALY_WINDOWS:
        is_anom |= ((df["timestamp"] >= s) & (df["timestamp"] <= e)).values
    df["is_true_anomaly"] = is_anom
    return df[["timestamp", "value", "is_true_anomaly"]]

df = load_nyc_taxi()
print(f"rows: {len(df)} | range: {df.timestamp.min()} → {df.timestamp.max()}")
print(f"anomaly rate: {df.is_true_anomaly.mean():.1%}")
df.head()


## 3. Load TimesFM 2.5

In [ ]:
CTX_LEN = 1024
torch.set_float32_matmul_precision("high")

model = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")
model.compile(
    timesfm.ForecastConfig(
        max_context=CTX_LEN,
        max_horizon=128,
        normalize_inputs=True,
        use_continuous_quantile_head=False,   # not needed for embedding extraction
        force_flip_invariance=True,
        infer_is_positive=True,
        fix_quantile_crossing=False,
    )
)
print("model loaded and compiled")
print("type(model.model):", type(model.model).__name__)


## 4. Auto-discover transformer layers and attach the hook

The exact internal path (`model.model.<...>.layers`) can change between versions. This helper finds the longest `nn.ModuleList` of transformer-like blocks and uses that as the stack.

If you want to inspect the structure yourself, just run `print(model.model)`.


In [ ]:
def find_transformer_stack(root: torch.nn.Module):
    '''Return (path_str, ModuleList) for the longest ModuleList of attention-bearing blocks.'''
    best = None
    for name, mod in root.named_modules():
        if isinstance(mod, torch.nn.ModuleList) and len(mod) >= 4:
            first = mod[0]
            attr_names = set(dir(first))
            if attr_names & {"self_attn", "attention", "attn", "qkv_proj", "q_proj"}:
                if best is None or len(mod) > len(best[1]):
                    best = (name, mod)
    if best is None:
        raise RuntimeError("Could not find a transformer stack. Run `print(model.model)` and pick a layer manually.")
    return best

stack_path, layers = find_transformer_stack(model.model)
N_LAYERS = len(layers)
LAYER_IDX = N_LAYERS // 2          # middle layer; sweep later
print(f"transformer stack at: model.model.{stack_path}  ({N_LAYERS} layers)")
print(f"hooking layer {LAYER_IDX}")

_cache = {}
def _hook(_m, _i, out):
    h = out[0] if isinstance(out, tuple) else out
    _cache["h"] = h.detach()

# Remove any previously registered hook before re-running this cell
if "_handle" in globals(): _handle.remove()
_handle = layers[LAYER_IDX].register_forward_hook(_hook)


## 5. Embed every window

We pass each window through `model.forecast(...)` — the hook fires during the forward pass and caches the chosen layer's hidden state. We mean-pool over the patch dimension to get one vector per window.


In [ ]:
STRIDE = 24                # 12-hour stride
BATCH  = 16

def make_windows(values, ctx_len=CTX_LEN, stride=STRIDE):
    xs, ends = [], []
    for end in range(ctx_len, len(values), stride):
        xs.append(values[end - ctx_len:end])
        ends.append(end - 1)
    return [np.asarray(x, dtype=np.float32) for x in xs], np.array(ends)

@torch.no_grad()
def embed_batch(windows_list):
    '''windows_list: list of 1-D np arrays. Runs them through model.forecast,
    catches the hooked hidden state, mean-pools per series.'''
    _ = model.forecast(horizon=1, inputs=windows_list)
    h = _cache["h"]                                  # [B, T_patches, D]
    return h.mean(dim=1).cpu().numpy()

def embed_all(values):
    w, ends = make_windows(values)
    out = []
    for i in range(0, len(w), BATCH):
        out.append(embed_batch(w[i:i+BATCH]))
    return np.vstack(out), ends

split = len(df) // 2
train_emb, _        = embed_all(df.value.values[:split])
all_emb,   all_ends = embed_all(df.value.values)
print("train_emb:", train_emb.shape, "| all_emb:", all_emb.shape)


## 6. Mahalanobis score

In [ ]:
def fit_gaussian(X, ridge=1e-3):
    mu = X.mean(axis=0)
    Xc = X - mu
    cov = (Xc.T @ Xc) / max(len(X) - 1, 1)
    cov += ridge * np.eye(cov.shape[0])
    return mu, np.linalg.inv(cov)

def mahalanobis(X, mu, cov_inv):
    Xc = X - mu
    return np.sqrt(np.einsum("ij,jk,ik->i", Xc, cov_inv, Xc))

mu, cov_inv = fit_gaussian(train_emb)
scores_w = mahalanobis(all_emb, mu, cov_inv)
print(f"score: min={scores_w.min():.2f}  median={np.median(scores_w):.2f}  max={scores_w.max():.2f}")


## 7. Map to time axis + plot

In [ ]:
def align(ends, scores, n):
    out = np.full(n, np.nan)
    out[ends] = scores
    last = np.nan
    for i in range(n):
        if not np.isnan(out[i]): last = out[i]
        elif not np.isnan(last): out[i] = last
    return out

scores_series = align(all_ends, scores_w, len(df))

fig, ax = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
ax[0].plot(df.timestamp, df.value, lw=0.4, color="steelblue")
ax[1].plot(df.timestamp, scores_series, lw=0.6, color="purple")
for a in ax:
    for s, e in NAB_ANOMALY_WINDOWS:
        a.axvspan(pd.Timestamp(s), pd.Timestamp(e), color="red", alpha=0.15)
ax[0].set_ylabel("passengers"); ax[1].set_ylabel("Mahalanobis"); ax[1].set_xlabel("time")
plt.tight_layout(); plt.show()


## 8. Evaluate vs. residual+IQR baseline

In [ ]:
def evaluate(flags, truth):
    flags = flags.astype(bool); truth = truth.astype(bool)
    tp = int((flags & truth).sum()); fp = int((flags & ~truth).sum()); fn = int((~flags & truth).sum())
    p = tp / max(tp+fp, 1); r = tp / max(tp+fn, 1)
    f1 = 2*p*r / max(p+r, 1e-9)
    return {"precision": round(p,3), "recall": round(r,3), "f1": round(f1,3),
            "tp": tp, "fp": fp, "fn": fn}

# Baseline: residual+IQR, 1-day rolling mean
roll = pd.Series(df.value.values).rolling(window=48, center=True, min_periods=12).mean()
residual = df.value.values - roll.values
m = np.isfinite(residual)
q1, q3 = np.percentile(residual[m], 25), np.percentile(residual[m], 75)
iqr_v = q3 - q1
flags_iqr = np.zeros(len(df), dtype=bool)
flags_iqr[m] = (residual[m] < q1 - 3*iqr_v) | (residual[m] > q3 + 3*iqr_v)

# Embedding score, threshold at anomaly-rate-matched percentile
rate = df.is_true_anomaly.mean()
pct = 100 * (1 - rate)
thr = np.percentile(scores_series[np.isfinite(scores_series)], pct)
flags_emb = np.isfinite(scores_series) & (scores_series >= thr)

print(f"anomaly rate: {rate:.1%} | embedding threshold @ {pct:.1f}th percentile")
print("residual + IQR :", evaluate(flags_iqr, df.is_true_anomaly.values))
print("embedding score:", evaluate(flags_emb, df.is_true_anomaly.values))


## 9. Per-event hit table

In [ ]:
def event_hits(flags, windows, ts):
    rows = []
    for s, e in windows:
        m = (ts >= pd.Timestamp(s)) & (ts <= pd.Timestamp(e))
        rows.append({"window": f"{s[:10]} → {e[:10]}",
                     "duration_pts": int(m.sum()),
                     "flagged_pts":  int((flags & m).sum()),
                     "hit":          bool((flags & m).any())})
    return pd.DataFrame(rows)

print("=== residual + IQR ===")
print(event_hits(flags_iqr, NAB_ANOMALY_WINDOWS, df.timestamp).to_string(index=False))
print("\n=== embedding score ===")
print(event_hits(flags_emb, NAB_ANOMALY_WINDOWS, df.timestamp).to_string(index=False))


## 10. If something breaks

- **`AttributeError: TimesFM_2p5_200M_torch`** → the `timesfm` package on PyPI is too old. Run cell 1 install line (GitHub source).
- **Hook never fires (`_cache` empty)** → run `print(model.model)` and find the actual transformer-stack attribute name. Replace `find_transformer_stack` with a direct path like `model.model.<path>.layers`.
- **OOM** → reduce `BATCH` to 8 or 4, or `CTX_LEN` to 512.

When this runs cleanly, send me the F1 numbers and the per-event hit table. Then we wire up Step 2 (conformal threshold).
